In [1]:
import pandas as pd
import numpy as np
import time

In [2]:
file_path = r'vectorize_optimize.csv'

df = pd.read_csv(file_path, encoding='utf-8')

df

,商品ID,成本(元),售价(元),销量,税率(%)
0,P001,1500,2999,20,13
1,P002,100,199,25,13
2,P003,80,159,18,13
3,P004,50,99,30,13
4,P005,180,299,22,13
5,P006,2000,3999,19,13
6,P007,90,199,28,13
7,P008,40,89,35,13
8,P009,1600,2999,40,13
9,P010,60,129,15,13


In [3]:
df = pd.concat([df]*10000, ignore_index=True)
print(f"测试数据量：{len(df)} 行")

测试数据量：100000 行


In [4]:
df

,商品ID,成本(元),售价(元),销量,税率(%)
0,P001,1500,2999,20,13
1,P002,100,199,25,13
2,P003,80,159,18,13
3,P004,50,99,30,13
4,P005,180,299,22,13
...,...,...,...,...,...
99995,P006,2000,3999,19,13
99996,P007,90,199,28,13
99997,P008,40,89,35,13
99998,P009,1600,2999,40,13


In [5]:
start_time = time.time()
# 初始化空列表存储结果
profit_single = []
profit_total = []
# 逐行循环计算（效率极低）
for i in range(len(df)):
    cost = df.iloc[i]['成本(元)']
    price = df.iloc[i]['售价(元)']
    tax = df.iloc[i]['税率(%)']
    sales = df.iloc[i]['销量']

    single = (price - cost) - price * tax / 100
    total = single * sales

    profit_single.append(single)
    profit_total.append(total)
# 赋值回DataFrame
df['单件利润_循环'] = profit_single
df['总利润_循环'] = profit_total

loop_time = time.time() - start_time
print(f"\n2. for循环计算：耗时 {loop_time:.4f} 秒")


2. for循环计算：耗时 9.3249 秒


In [6]:
df

,商品ID,成本(元),售价(元),销量,税率(%),单件利润_循环,总利润_循环
0,P001,1500,2999,20,13,1109.13,22182.60
1,P002,100,199,25,13,73.13,1828.25
2,P003,80,159,18,13,58.33,1049.94
3,P004,50,99,30,13,36.13,1083.90
4,P005,180,299,22,13,80.13,1762.86
...,...,...,...,...,...,...,...
99995,P006,2000,3999,19,13,1479.13,28103.47
99996,P007,90,199,28,13,83.13,2327.64
99997,P008,40,89,35,13,37.43,1310.05
99998,P009,1600,2999,40,13,1009.13,40365.20


In [7]:
start_time = time.time()
profit_single_iter = []
profit_total_iter = []
# iterrows逐行迭代
for idx, row in df.iterrows():
    single = (row['售价(元)'] - row['成本(元)']) - row['售价(元)'] * row['税率(%)'] / 100
    total = single * row['销量']
    profit_single_iter.append(single)
    profit_total_iter.append(total)

df['单件利润_iterrows'] = profit_single_iter
df['总利润_iterrows'] = profit_total_iter

iterrows_time = time.time() - start_time
print(f"3. iterrows计算：耗时 {iterrows_time:.4f} 秒（比循环快 {(loop_time-iterrows_time)/loop_time*100:.2f}%）")

3. iterrows计算：耗时 1.9975 秒（比循环快 78.58%）


In [8]:
start_time = time.time()
# 整列批量运算（无循环，Pandas内置优化）
df['单件利润_向量化'] = (df['售价(元)'] - df['成本(元)']) - df['售价(元)'] * df['税率(%)'] / 100
df['总利润_向量化'] = df['单件利润_向量化'] * df['销量']

vector_time = time.time() - start_time
speed_up = loop_time / vector_time  # 提速倍数
print(f"4. 向量化操作：耗时 {vector_time:.4f} 秒（比for循环快 {speed_up:.1f} 倍！）")

4. 向量化操作：耗时 0.0028 秒（比for循环快 3350.9 倍！）


In [9]:
df

,商品ID,成本(元),售价(元),销量,税率(%),单件利润_循环,总利润_循环,单件利润_iterrows,总利润_iterrows,单件利润_向量化,总利润_向量化
0,P001,1500,2999,20,13,1109.13,22182.60,1109.13,22182.60,1109.13,22182.60
1,P002,100,199,25,13,73.13,1828.25,73.13,1828.25,73.13,1828.25
2,P003,80,159,18,13,58.33,1049.94,58.33,1049.94,58.33,1049.94
3,P004,50,99,30,13,36.13,1083.90,36.13,1083.90,36.13,1083.90
4,P005,180,299,22,13,80.13,1762.86,80.13,1762.86,80.13,1762.86
...,...,...,...,...,...,...,...,...,...,...,...
99995,P006,2000,3999,19,13,1479.13,28103.47,1479.13,28103.47,1479.13,28103.47
99996,P007,90,199,28,13,83.13,2327.64,83.13,2327.64,83.13,2327.64
99997,P008,40,89,35,13,37.43,1310.05,37.43,1310.05,37.43,1310.05
99998,P009,1600,2999,40,13,1009.13,40365.20,1009.13,40365.20,1009.13,40365.20


In [19]:
start_time = time.time()
# 转换为NumPy数组计算（底层C实现，速度更快）
cost_np = df['成本(元)'].values
price_np = df['售价(元)'].values
tax_np = df['税率(%)'].values
sales_np = df['销量'].values

df['单件利润_NumPy'] = (price_np - cost_np) - price_np * tax_np / 100
df['总利润_NumPy'] = df['单件利润_NumPy'] * sales_np

numpy_time = time.time() - start_time
speed_up_numpy = loop_time / numpy_time
print(f"5. NumPy向量化：耗时 {numpy_time:.4f} 秒（比for循环快 {speed_up_numpy:.1f} 倍！）")

5. NumPy向量化：耗时 0.0019 秒（比for循环快 4957.1 倍！）


In [20]:
print("\n6. 计算结果一致性验证：")
# 对比循环和向量化结果（浮点精度允许微小误差）
is_same = np.isclose(df['总利润_循环'], df['总利润_向量化']).all()
print(f"循环 vs 向量化 结果一致：{is_same}")
is_same_numpy = np.isclose(df['总利润_向量化'], df['总利润_NumPy']).all()
print(f"Pandas向量化 vs NumPy向量化 结果一致：{is_same_numpy}")


6. 计算结果一致性验证：
循环 vs 向量化 结果一致：True
Pandas向量化 vs NumPy向量化 结果一致：True


In [21]:
df

,商品ID,成本(元),售价(元),销量,税率(%),单件利润_循环,总利润_循环,单件利润_iterrows,总利润_iterrows,单件利润_向量化,总利润_向量化,单件利润_NumPy,总利润_NumPy
0,P001,1500,2999,20,13,1109.13,22182.60,1109.13,22182.60,1109.13,22182.60,1109.13,22182.60
1,P002,100,199,25,13,73.13,1828.25,73.13,1828.25,73.13,1828.25,73.13,1828.25
2,P003,80,159,18,13,58.33,1049.94,58.33,1049.94,58.33,1049.94,58.33,1049.94
3,P004,50,99,30,13,36.13,1083.90,36.13,1083.90,36.13,1083.90,36.13,1083.90
4,P005,180,299,22,13,80.13,1762.86,80.13,1762.86,80.13,1762.86,80.13,1762.86
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,P006,2000,3999,19,13,1479.13,28103.47,1479.13,28103.47,1479.13,28103.47,1479.13,28103.47
99996,P007,90,199,28,13,83.13,2327.64,83.13,2327.64,83.13,2327.64,83.13,2327.64
99997,P008,40,89,35,13,37.43,1310.05,37.43,1310.05,37.43,1310.05,37.43,1310.05
99998,P009,1600,2999,40,13,1009.13,40365.20,1009.13,40365.20,1009.13,40365.20,1009.13,40365.20


In [22]:
def calc_profit(row):
    single = (row['售价(元)'] - row['成本(元)']) - row['售价(元)'] * row['税率(%)'] / 100
    total = single * row['销量']
    return pd.Series([single, total])

start_time = time.time()
df[['单件利润_apply', '总利润_apply']] = df.apply(calc_profit, axis=1)
apply_time = time.time() - start_time
print(f"\n7. apply向量化：耗时 {apply_time:.4f} 秒（比循环快 {loop_time/apply_time:.1f} 倍）")


7. apply向量化：耗时 5.5499 秒（比循环快 1.7 倍）


In [23]:
optimize_summary = pd.DataFrame(
    {'运算方式': ['for循环', 'iterrows', 'Pandas向量化', 'NumPy向量化', 'apply向量化'],
     '耗时(秒)': [loop_time, iterrows_time, vector_time, numpy_time, apply_time],
     '相对速度(倍)': [1, loop_time/iterrows_time, loop_time/vector_time, loop_time/numpy_time, loop_time/apply_time]
     }
).round(2)
print("\n8. 优化效果汇总：")
print(optimize_summary)


8. 优化效果汇总：
        运算方式  耗时(秒)  相对速度(倍)
0      for循环   9.32     1.00
1   iterrows   2.00     4.67
2  Pandas向量化   0.00  3350.87
3   NumPy向量化   0.00  4957.07
4   apply向量化   5.55     1.68
